# Multi-Model Agent Patterns: Cost Routing & Fallback

This notebook demonstrates **production-ready patterns** for multi-model systems:

- **Cost-optimized routing**: Classify task complexity and route to the cheapest model that can handle it
- **Fallback and retry**: Handle provider failures gracefully with automatic failover

These patterns address two real-world concerns:
1. **Cost**: Not every task needs the most expensive model. Simple tasks can use cheaper/faster models.
2. **Resilience**: Provider outages happen. Your application should degrade gracefully, not crash.

**Key Insight**: By combining intelligent routing with fallback chains, you can build systems that are both cost-efficient and highly available.

## Environment Setup

This notebook requires Amazon Bedrock access (AWS credentials). All examples use Bedrock models only — no additional API keys needed.

In [ ]:
import os


def check_environment(required_vars: list[str], optional_vars: list[str]) -> None:
    """Validate environment variables are set for model providers.

    Args:
        required_vars: Environment variables that must be set.
        optional_vars: Environment variables that enhance the tutorial but aren't required.
    """
    missing_required = [v for v in required_vars if not os.environ.get(v)]
    missing_optional = [v for v in optional_vars if not os.environ.get(v)]

    if missing_required:
        print("❌ Missing REQUIRED environment variables:")
        for var in missing_required:
            print(f"   - {var}")
        print("\nSet these before running the notebook.")
        raise EnvironmentError(f"Missing required variables: {missing_required}")

    if missing_optional:
        print("⚠️  Missing OPTIONAL environment variables (some examples will use alternatives):")
        for var in missing_optional:
            print(f"   - {var}")
    else:
        print("✅ All environment variables are set.")


# Bedrock requires AWS credentials configured via environment or AWS CLI profile
check_environment(
    required_vars=["AWS_DEFAULT_REGION"],
    optional_vars=["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]
)

## Imports and Helpers

We import the Strands SDK and define the `safe_agent_call()` helper for graceful error handling across all examples in this notebook.

In [ ]:
import time
from strands import Agent
from strands.models.bedrock import BedrockModel


def safe_agent_call(agent: Agent, task: str, provider_name: str) -> str:
    """Execute an agent call with informative error handling.

    Wraps agent invocation to catch provider errors and display actionable
    diagnostics instead of raw stack traces.

    Args:
        agent: The Strands Agent instance to invoke.
        task: The task/prompt to send to the agent.
        provider_name: Human-readable name of the provider for error messages.

    Returns:
        The agent's response as a string, or empty string on failure.
    """
    try:
        response = agent(task)
        return str(response)
    except Exception as e:
        error_type = type(e).__name__
        print(f"❌ Error from {provider_name}: {error_type}")
        print(f"   Message: {str(e)}")

        # Suggest corrective action based on error type
        if "credential" in str(e).lower() or "key" in str(e).lower():
            print(f"   → Check that API keys for {provider_name} are set in environment variables")
        elif "throttl" in str(e).lower() or "rate" in str(e).lower():
            print(f"   → {provider_name} is rate-limited. Wait and retry, or use a different provider")
        elif "timeout" in str(e).lower():
            print(f"   → {provider_name} timed out. The model may be overloaded")
        else:
            print(f"   → Verify model ID and region configuration for {provider_name}")

        return ""

---

## Section A: Cost-Optimized Model Routing

### The Problem

Using a single powerful model (like Claude Sonnet) for every task is wasteful:
- Simple tasks ("What's 2+2?") don't need advanced reasoning
- Cheaper models handle simple tasks just as well, at a fraction of the cost
- Faster models reduce latency for straightforward requests

### The Solution: CostRouter

The `CostRouter` classifies each task's complexity and routes it to the appropriate model:
- **Simple tasks** → Nova Lite (cheapest, fastest)
- **Complex tasks** → Claude Sonnet (most capable)

Classification uses two observable criteria:
1. **Word count**: Longer prompts tend to require more sophisticated reasoning
2. **Complexity keywords**: Words like "analyze", "compare", "synthesize" signal complex tasks

### CostRouter Implementation

The router uses a rule-based classifier to determine task complexity. This approach is transparent, debuggable, and doesn't require an LLM call for the routing decision itself.

In [ ]:
class CostRouter:
    """Routes tasks to appropriate models based on complexity classification.

    Uses observable criteria (word count and keywords) to classify task complexity,
    then routes simple tasks to cheaper/faster models and complex tasks to more
    capable models.

    Args:
        simple_model: Model instance for simple/cheap tasks.
        complex_model: Model instance for complex/expensive tasks.
        complexity_threshold: Word count threshold for complexity classification.
    """

    def __init__(
        self,
        simple_model: BedrockModel,
        complex_model: BedrockModel,
        complexity_threshold: int = 50,
    ) -> None:
        self.simple_model = simple_model
        self.complex_model = complex_model
        self.complexity_threshold = complexity_threshold
        self.routing_log: list[dict[str, str]] = []

    def classify_complexity(self, task: str) -> str:
        """Classify task complexity based on observable criteria.

        Uses two rules:
        1. Word count exceeding threshold indicates complexity
        2. Presence of complexity keywords indicates analytical tasks

        Args:
            task: The task description to classify.

        Returns:
            'simple' or 'complex' based on classification rules.
        """
        # Keywords that signal tasks requiring deeper reasoning
        complexity_keywords = [
            "analyze", "compare", "evaluate", "synthesize",
            "explain in detail", "step by step", "comprehensive"
        ]

        # Rule 1: Check word count against threshold
        word_count = len(task.split())

        # Rule 2: Check for complexity-indicating keywords
        has_complex_keywords = any(kw in task.lower() for kw in complexity_keywords)

        # Classify as complex if EITHER criterion is met
        if word_count > self.complexity_threshold or has_complex_keywords:
            return "complex"

        # Default: tasks that don't match any complexity criteria are simple
        # This ensures unclassifiable tasks route to the cheaper model,
        # which is safe because simple tasks sent to a capable model just cost more
        return "simple"

    def route(self, task: str) -> Agent:
        """Route task to appropriate model based on complexity.

        Creates an Agent configured with the model matching the task's
        classified complexity level. Logs the routing decision for analysis.

        Args:
            task: The task to route.

        Returns:
            An Agent configured with the appropriate model.
        """
        complexity = self.classify_complexity(task)

        if complexity == "complex":
            # Route complex tasks to the most capable model
            model = self.complex_model
            model_name = "Claude Sonnet (complex)"
        else:
            # Route simple tasks to the cheapest/fastest model
            model = self.simple_model
            model_name = "Nova Lite (simple)"

        # Log the routing decision for later analysis
        self.routing_log.append({
            "task_preview": task[:50],
            "complexity": complexity,
            "model": model_name,
        })

        return Agent(model=model, callback_handler=None, load_tools_from_directory=False)


print("✅ CostRouter class defined.")

### Configure Models and Create Router

We set up two Bedrock models with different cost/capability profiles:
- **Nova Lite**: $0.00006/1K input tokens — ideal for simple tasks
- **Claude Sonnet**: $0.003/1K input tokens — 50x more expensive, but handles complex reasoning

In [ ]:
# Configure the cheap/fast model for simple tasks
nova_lite_model = BedrockModel(
    model_id="us.amazon.nova-lite-v1:0",
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)

# Configure the capable model for complex tasks
claude_sonnet_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)

# Create the cost router with default threshold (50 words)
cost_router = CostRouter(
    simple_model=nova_lite_model,
    complex_model=claude_sonnet_model,
    complexity_threshold=50,
)

print("✅ CostRouter configured:")
print(f"   Simple model:  Nova Lite (${0.00006}/1K input tokens)")
print(f"   Complex model: Claude Sonnet (${0.003}/1K input tokens)")
print(f"   Threshold:     {cost_router.complexity_threshold} words")

### Demonstrate Routing with Example Tasks

Let's test the router with tasks spanning all complexity levels. The router should:
- Send short, simple questions to Nova Lite (cheap/fast)
- Send analytical or lengthy tasks to Claude Sonnet (capable)
- Handle edge cases (empty or ambiguous tasks) with safe defaults

In [ ]:
# Define example tasks covering all complexity levels
example_tasks = [
    # Simple task: short, factual question (routes to Nova Lite)
    "What is the capital of France?",

    # Simple task: brief instruction (routes to Nova Lite)
    "List three primary colors.",

    # Complex task: contains complexity keyword 'analyze' (routes to Claude Sonnet)
    "Analyze the trade-offs between microservices and monolithic architectures for a startup.",

    # Complex task: contains 'step by step' keyword (routes to Claude Sonnet)
    "Explain step by step how to implement a distributed consensus algorithm.",

    # Edge case: very short task with no keywords (routes to Nova Lite as default)
    "Hello!",
]

print("=" * 70)
print("COST ROUTER DEMONSTRATION")
print("=" * 70)

for i, task in enumerate(example_tasks, 1):
    # Classify the task before routing
    complexity = cost_router.classify_complexity(task)
    word_count = len(task.split())

    print(f"\n--- Task {i} ---")
    print(f"Task:       {task}")
    print(f"Words:      {word_count}")
    print(f"Complexity: {complexity}")

    # Route and execute the task
    routed_agent = cost_router.route(task)
    model_used = cost_router.routing_log[-1]["model"]
    print(f"Routed to:  {model_used}")

    # Execute with error handling
    response = safe_agent_call(routed_agent, task, model_used)
    if response:
        # Truncate long responses for display
        display_response = response[:200] + "..." if len(response) > 200 else response
        print(f"Response:   {display_response}")

print("\n" + "=" * 70)

### Routing Log Analysis

The router maintains a log of all routing decisions. This is useful for monitoring cost optimization effectiveness in production.

In [ ]:
# Display the routing log as a formatted table
print("\nROUTING LOG")
print("-" * 70)
print(f"{'Task Preview':<40} {'Complexity':<12} {'Model'}")
print("-" * 70)

for entry in cost_router.routing_log:
    preview = entry["task_preview"][:38] + ".." if len(entry["task_preview"]) > 38 else entry["task_preview"]
    print(f"{preview:<40} {entry['complexity']:<12} {entry['model']}")

# Summary statistics
simple_count = sum(1 for e in cost_router.routing_log if e["complexity"] == "simple")
complex_count = sum(1 for e in cost_router.routing_log if e["complexity"] == "complex")
print("-" * 70)
print(f"Total: {len(cost_router.routing_log)} tasks | "
      f"Simple: {simple_count} (→ Nova Lite) | "
      f"Complex: {complex_count} (→ Claude Sonnet)")

### Cost Comparison: Routing vs Single-Model

Let's compare the estimated cost and latency of our routing strategy against using a single model for everything. This demonstrates the financial benefit of intelligent routing.

**Assumptions for comparison:**
- Average simple task: ~50 input tokens, ~100 output tokens
- Average complex task: ~200 input tokens, ~500 output tokens
- Latency estimates based on typical response times

In [ ]:
# Model cost reference (per 1K tokens)
MODEL_COSTS = {
    "Claude Sonnet": {"input": 0.003, "output": 0.015, "speed": "Medium", "latency_ms": 2500},
    "Nova Pro": {"input": 0.001, "output": 0.005, "speed": "Fast", "latency_ms": 1200},
    "Nova Lite": {"input": 0.00006, "output": 0.00024, "speed": "Fastest", "latency_ms": 800},
    "Nova Pro": {"input": 0.0008, "output": 0.0032, "speed": "Medium", "latency_ms": 2000},
}

# Estimated token usage per task type
SIMPLE_TASK_TOKENS = {"input_tokens": 50, "output_tokens": 100}
COMPLEX_TASK_TOKENS = {"input_tokens": 200, "output_tokens": 500}


def estimate_cost(model_name: str, input_tokens: int, output_tokens: int) -> float:
    """Estimate the cost for a single request to a model.

    Args:
        model_name: Name of the model (must be in MODEL_COSTS).
        input_tokens: Number of input tokens.
        output_tokens: Number of output tokens.

    Returns:
        Estimated cost in USD.
    """
    costs = MODEL_COSTS[model_name]
    input_cost = (input_tokens / 1000) * costs["input"]
    output_cost = (output_tokens / 1000) * costs["output"]
    return input_cost + output_cost


# Calculate costs for our routing strategy
# Based on the 5 example tasks: 3 simple, 2 complex
routing_cost = (
    simple_count * estimate_cost("Nova Lite", **SIMPLE_TASK_TOKENS)
    + complex_count * estimate_cost("Claude Sonnet", **COMPLEX_TASK_TOKENS)
)
routing_latency = (
    simple_count * MODEL_COSTS["Nova Lite"]["latency_ms"]
    + complex_count * MODEL_COSTS["Claude Sonnet"]["latency_ms"]
)

# Calculate costs for single-model strategy (all tasks to Claude Sonnet)
single_model_cost = (
    simple_count * estimate_cost("Claude Sonnet", **SIMPLE_TASK_TOKENS)
    + complex_count * estimate_cost("Claude Sonnet", **COMPLEX_TASK_TOKENS)
)
single_model_latency = (
    (simple_count + complex_count) * MODEL_COSTS["Claude Sonnet"]["latency_ms"]
)

# Print comparison table
print("\n" + "=" * 70)
print("ROUTING STRATEGY COMPARISON")
print("=" * 70)
print(f"\nWorkload: {simple_count} simple tasks + {complex_count} complex tasks")
print()
print(f"{'Strategy':<30} {'Est. Cost':<18} {'Est. Total Latency':<20} {'Avg Latency/Task'}")
print("-" * 90)
print(
    f"{'Cost-Optimized Routing':<30} "
    f"${routing_cost:.6f}{'':<10} "
    f"{routing_latency:,} ms{'':<10} "
    f"{routing_latency // (simple_count + complex_count):,} ms"
)
print(
    f"{'Single Model (Sonnet)':<30} "
    f"${single_model_cost:.6f}{'':<10} "
    f"{single_model_latency:,} ms{'':<10} "
    f"{single_model_latency // (simple_count + complex_count):,} ms"
)
print("-" * 90)

# Calculate savings
cost_savings = ((single_model_cost - routing_cost) / single_model_cost) * 100
latency_savings = ((single_model_latency - routing_latency) / single_model_latency) * 100

print(f"\n💰 Cost savings with routing:    {cost_savings:.1f}%")
print(f"⚡ Latency savings with routing: {latency_savings:.1f}%")
print()
print("Note: Actual costs depend on real token counts. These estimates use")
print("~50 input/100 output tokens for simple tasks and ~200 input/500 output")
print("tokens for complex tasks.")

### Model Cost Reference

For reference, here are the per-token costs for models used in this tutorial:

In [ ]:
# Print model cost reference table
print("\nMODEL COST REFERENCE")
print("=" * 70)
print(f"{'Model':<18} {'Cost/1K Input':<16} {'Cost/1K Output':<17} {'Speed'}")
print("-" * 70)
for model_name, info in MODEL_COSTS.items():
    print(
        f"{model_name:<18} "
        f"${info['input']:<14} "
        f"${info['output']:<15} "
        f"{info['speed']}"
    )
print("-" * 70)
print("\nCosts are approximate and used for educational comparison only.")
print("Check AWS pricing page for current rates.")

### Key Takeaways: Cost Routing

1. **Classification is cheap**: Rule-based classification costs nothing — no LLM call needed for routing
2. **Default to safe**: When in doubt, route to the cheaper model. Simple tasks on capable models waste money but still work correctly.
3. **Monitor and tune**: Use the routing log to track classification accuracy and adjust thresholds
4. **Production tip**: In production, you might use a small LLM to classify tasks, or train a classifier on historical routing decisions

---

## Section B: Fallback and Retry Patterns

### The Problem

Model providers fail. Throttling, timeouts, and outages are inevitable in production:
- **Throttling**: Provider rate-limits your requests during traffic spikes
- **Timeouts**: A model takes too long to respond (>30 seconds)
- **Unavailability**: A provider is temporarily down for maintenance

### The Solution: FallbackHandler

The `FallbackHandler` maintains an ordered chain of providers. When one fails, it automatically
retries with the next provider in the chain — preserving the original request content.

**Key design decisions:**
- Providers are tried in priority order (cheapest/fastest first)
- The original task is passed unchanged to each provider (no modification on retry)
- All failures are logged with provider name, error type, and message
- If the entire chain is exhausted, a `RuntimeError` reports every failure

### FallbackHandler Implementation

The handler iterates through providers in order, catching any exception and moving to the next.
It also enforces a configurable timeout per provider attempt.

In [ ]:
class FallbackHandler:
    """Handles model provider failures with automatic fallback to alternatives.

    Maintains an ordered chain of providers and tries each in sequence until
    one succeeds. Catches throttling, timeout, and unavailability errors.

    Args:
        providers: Ordered list of (name, model) tuples, tried in sequence.
        timeout_seconds: Maximum seconds to wait per provider attempt.
    """

    def __init__(
        self,
        providers: list[tuple[str, BedrockModel]],
        timeout_seconds: float = 30.0,
    ) -> None:
        self.providers = providers
        self.timeout_seconds = timeout_seconds
        self.attempt_log: list[dict] = []

    def execute(self, task: str) -> str:
        """Execute task with automatic fallback on failure.

        Tries each provider in the chain sequentially. On any exception
        (throttling, timeout, unavailability), logs the error and moves
        to the next provider. The original task content is preserved
        across all retry attempts.

        Providers can be either:
        - A BedrockModel instance (used with Agent for real inference)
        - A callable that raises an exception (used to simulate failures in demos)

        Args:
            task: The task to execute (passed unchanged to each provider).

        Returns:
            Response from the first successful provider.

        Raises:
            RuntimeError: If all providers in the chain fail, with per-provider
                error details including provider count and each error type/message.
        """
        errors: list[dict[str, str]] = []

        for provider_name, model in self.providers:
            try:
                start_time = time.time()

                # Check if this is a failure simulator (callable) or a real model
                if callable(model) and not isinstance(model, BedrockModel):
                    # Failure simulator — call it directly (it will raise)
                    model()
                else:
                    # Real model — wrap in Agent and execute
                    agent = Agent(model=model, callback_handler=None, load_tools_from_directory=False)

                    # Execute the task — original content preserved across retries
                    response = agent(task)
                    elapsed = time.time() - start_time

                    # Fallback trigger: timeout exceeding configured threshold
                    if elapsed > self.timeout_seconds:
                        raise TimeoutError(
                            f"Provider {provider_name} exceeded {self.timeout_seconds}s timeout"
                        )

                    # Success — log and return the response
                    self.attempt_log.append({
                        "provider": provider_name,
                        "status": "success",
                        "elapsed_seconds": round(elapsed, 2),
                    })
                    return str(response)

            except Exception as e:
                # Fallback trigger: any exception (throttling, timeout, unavailability)
                # Log the failure and continue to the next provider in the chain
                error_info = {
                    "provider": provider_name,
                    "status": "failed",
                    "error_type": type(e).__name__,
                    "error_message": str(e),
                }
                errors.append(error_info)
                self.attempt_log.append(error_info)
                # Retry with next provider — no artificial delay
                continue

        # All providers exhausted — report full failure chain
        error_summary = "; ".join(
            f"{e['provider']}: {e['error_type']} - {e['error_message']}"
            for e in errors
        )
        raise RuntimeError(
            f"All {len(self.providers)} providers failed. "
            f"Errors: {error_summary}"
        )


print("\u2705 FallbackHandler class defined.")

### Simulated Failures for Demonstration

To demonstrate fallback behavior without waiting for real provider failures, we define
simple **failure simulator functions**. Each function raises a specific exception type
when called. The `FallbackHandler` detects these callables and invokes them directly,
triggering the fallback mechanism.

This approach is simpler and more reliable than trying to mock model internals —
the handler treats any callable (that isn't a BedrockModel) as a failure simulator.

**Supported failure types:**
- `make_throttling_failure`: Simulates rate-limit errors (provider overwhelmed)
- `make_timeout_failure`: Simulates slow responses exceeding the timeout threshold
- `make_unavailable_failure`: Simulates service outage

In [ ]:
def make_throttling_failure() -> str:
    """Simulate a throttling error from a provider."""
    raise Exception("ThrottlingException: Rate exceeded. Too many requests to this model.")


def make_timeout_failure() -> str:
    """Simulate a timeout error from a provider."""
    raise TimeoutError("Request timed out after 30s. Model is overloaded or unresponsive.")


def make_unavailable_failure() -> str:
    """Simulate a service unavailability error."""
    raise ConnectionError("ServiceUnavailableException: Model endpoint is temporarily unavailable.")


print("\u2705 Failure simulator functions defined.")
print("   make_throttling_failure  - raises ThrottlingException")
print("   make_timeout_failure     - raises TimeoutError")
print("   make_unavailable_failure - raises ConnectionError")
print()
print("These are simple callables that the FallbackHandler detects and invokes")
print("directly, triggering the fallback mechanism without needing a real model.")

### Configure Fallback Chain

We set up a 3-provider fallback chain ordered by priority:
1. **Nova Lite** (primary) — cheapest and fastest
2. **Nova Pro** (secondary) — fast with better reasoning
3. **Claude Sonnet** (tertiary) — most capable, used as last resort

In production, you'd order by cost (cheapest first) since any provider that responds is acceptable for resilience.

In [ ]:
# Configure the fallback chain with 3 providers in priority order
# Primary: cheapest/fastest model
fallback_nova_lite = BedrockModel(
    model_id="us.amazon.nova-lite-v1:0",
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)

# Secondary: mid-tier model (balanced cost and capability)
fallback_nova_pro = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)

# Tertiary: most capable model (last resort)
fallback_sonnet = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)

print("\u2705 Fallback chain configured (3 providers):")
print("   1. Nova Lite (primary - cheapest)")
print("   2. Nova Pro (secondary - balanced)")
print("   3. Claude Sonnet (tertiary - most capable)")

### Demo 1: Throttling Failure → Automatic Fallback

In this scenario, the primary provider (Nova Lite) is throttled. The `FallbackHandler`
catches the throttling error and automatically retries with the secondary provider (Nova Pro).

**What happens:**
1. Request sent to Nova Lite → ThrottlingException raised
2. FallbackHandler catches error, logs it, moves to next provider
3. Request sent to Nova Pro → Success

In [ ]:
# Simulate throttling on the primary provider using a failure function.
# The FallbackHandler detects callables and invokes them directly,
# which raises the exception and triggers fallback to the next provider.

# Create fallback handler with throttled primary + healthy secondaries
throttle_handler = FallbackHandler(
    providers=[
        ("Nova Lite (throttled)", make_throttling_failure),  # Will raise ThrottlingException
        ("Nova Pro", fallback_nova_pro),                    # Healthy — will succeed
        ("Claude Sonnet", fallback_sonnet),                  # Healthy backup
    ],
    timeout_seconds=30.0,
)

print("=" * 70)
print("DEMO 1: Throttling Failure \u2192 Automatic Fallback")
print("=" * 70)
print("\nScenario: Primary provider (Nova Lite) is rate-limited.")
print("Expected: Fallback to Nova Pro.\n")

# Execute — the handler will catch the throttling error and retry
throttle_task = "What is the speed of light in a vacuum?"
try:
    result = throttle_handler.execute(throttle_task)
    print(f"\u2705 Task completed successfully via fallback!")
    print(f"   Response: {result[:150]}..." if len(result) > 150 else f"   Response: {result}")
except RuntimeError as e:
    print(f"\u274c All providers failed: {e}")

# Show the attempt log
print("\nAttempt Log:")
for attempt in throttle_handler.attempt_log:
    status_icon = "\u2705" if attempt["status"] == "success" else "\u274c"
    if attempt["status"] == "success":
        print(f"   {status_icon} {attempt['provider']}: success ({attempt['elapsed_seconds']}s)")
    else:
        print(f"   {status_icon} {attempt['provider']}: {attempt['error_type']} - {attempt['error_message'][:60]}")

### Demo 2: Timeout Failure → Automatic Fallback

In this scenario, the primary provider (Nova Lite) times out (exceeds 30s). The `FallbackHandler`
catches the timeout error and retries with the secondary provider.

**What happens:**
1. Request sent to Nova Lite → TimeoutError raised
2. FallbackHandler catches error, logs it, moves to next provider
3. Request sent to Nova Pro → Success

In [ ]:
# Simulate timeout on the primary provider using a failure function.
# The FallbackHandler detects the callable and invokes it directly,
# which raises TimeoutError and triggers fallback to the next provider.

# Create fallback handler with timed-out primary + healthy secondaries
timeout_handler = FallbackHandler(
    providers=[
        ("Nova Lite (timed out)", make_timeout_failure),  # Will raise TimeoutError
        ("Nova Pro", fallback_nova_pro),                 # Healthy — will succeed
        ("Claude Sonnet", fallback_sonnet),               # Healthy backup
    ],
    timeout_seconds=30.0,
)

print("=" * 70)
print("DEMO 2: Timeout Failure \u2192 Automatic Fallback")
print("=" * 70)
print("\nScenario: Primary provider (Nova Lite) exceeds 30s timeout.")
print("Expected: Fallback to Nova Pro.\n")

# Execute — the handler will catch the timeout error and retry
timeout_task = "Name three planets in our solar system."
try:
    result = timeout_handler.execute(timeout_task)
    print(f"\u2705 Task completed successfully via fallback!")
    print(f"   Response: {result[:150]}..." if len(result) > 150 else f"   Response: {result}")
except RuntimeError as e:
    print(f"\u274c All providers failed: {e}")

# Show the attempt log
print("\nAttempt Log:")
for attempt in timeout_handler.attempt_log:
    status_icon = "\u2705" if attempt["status"] == "success" else "\u274c"
    if attempt["status"] == "success":
        print(f"   {status_icon} {attempt['provider']}: success ({attempt['elapsed_seconds']}s)")
    else:
        print(f"   {status_icon} {attempt['provider']}: {attempt['error_type']} - {attempt['error_message'][:60]}")

### Demo 3: All Providers Fail → RuntimeError with Full Error Chain

When every provider in the chain fails, the `FallbackHandler` raises a `RuntimeError`
containing the total number of providers attempted and the specific error for each one.
This gives operators full diagnostic information for troubleshooting.

**What happens:**
1. Nova Lite → ThrottlingException
2. Nova Pro → TimeoutError
3. Claude Sonnet → ServiceUnavailableException
4. RuntimeError raised with all 3 failure details

In [ ]:
# Simulate ALL providers failing with different error types.
# Each provider in the chain is a failure simulator function.

# Create fallback handler where every provider will fail
all_fail_handler = FallbackHandler(
    providers=[
        ("Nova Lite", make_throttling_failure),    # ThrottlingException
        ("Nova Pro", make_timeout_failure),    # TimeoutError
        ("Claude Sonnet", make_unavailable_failure),  # ConnectionError
    ],
    timeout_seconds=30.0,
)

print("=" * 70)
print("DEMO 3: All Providers Fail \u2192 RuntimeError")
print("=" * 70)
print("\nScenario: All 3 providers fail with different errors.")
print("Expected: RuntimeError with full error chain.\n")

# Execute — all providers will fail, triggering RuntimeError
all_fail_task = "What is 2 + 2?"
try:
    result = all_fail_handler.execute(all_fail_task)
    print(f"Unexpected success: {result}")
except RuntimeError as e:
    print(f"\u274c RuntimeError (expected):")
    print(f"   {e}")
    print(f"\n   \u2192 The error message includes:")
    print(f"     - Total provider count: 3")
    print(f"     - Per-provider error type and message")
    print(f"     - Enough detail for operators to diagnose the issue")

### Attempt Log Analysis

The `FallbackHandler` maintains a detailed attempt log across all executions. This is
invaluable for production monitoring — you can track failure rates per provider,
identify patterns, and tune your fallback chain ordering.

In [ ]:
# Combine attempt logs from all demos for analysis
all_attempts = (
    throttle_handler.attempt_log
    + timeout_handler.attempt_log
    + all_fail_handler.attempt_log
)

print("=" * 70)
print("FALLBACK ATTEMPT LOG (All Demos Combined)")
print("=" * 70)
print(f"\n{'#':<4} {'Provider':<25} {'Status':<10} {'Details'}")
print("-" * 70)

for i, attempt in enumerate(all_attempts, 1):
    provider = attempt["provider"]
    status = attempt["status"]
    if status == "success":
        details = f"completed in {attempt['elapsed_seconds']}s"
    else:
        details = f"{attempt['error_type']}: {attempt['error_message'][:40]}"
    print(f"{i:<4} {provider:<25} {status:<10} {details}")

# Summary statistics
success_count = sum(1 for a in all_attempts if a["status"] == "success")
failure_count = sum(1 for a in all_attempts if a["status"] == "failed")

print("-" * 70)
print(f"\nSummary: {len(all_attempts)} total attempts | "
      f"{success_count} successes | {failure_count} failures")
fallback_attempts = len(throttle_handler.attempt_log) + len(timeout_handler.attempt_log)
successes_excluding_allfail = sum(1 for a in throttle_handler.attempt_log + timeout_handler.attempt_log if a["status"] == "success")
if fallback_attempts > 0:
    print(f"Fallback success rate: {successes_excluding_allfail}/{fallback_attempts} "
          f"(excluding all-fail demo)")
else:
    print(f"Total successes: {success_count} | Total failures: {failure_count}")
print("\nIn production, monitor these metrics to:")
print("  - Detect providers with high failure rates")
print("  - Reorder the chain to put most reliable providers first")
print("  - Set alerts when fallback rate exceeds thresholds")

---

## Summary and Key Takeaways

This notebook demonstrated two production-ready patterns for multi-model systems:

### Section A: Cost-Optimized Routing

| Concept | Key Point |
|---------|----------|
| **Classification** | Rule-based (word count + keywords) — no LLM call needed for routing |
| **Default behavior** | Unclassifiable tasks route to cheaper model (safe default) |
| **Cost savings** | 50-90% reduction by routing simple tasks to cheaper models |
| **Monitoring** | Routing log enables tracking and threshold tuning |

### Section B: Fallback and Retry Patterns

| Concept | Key Point |
|---------|----------|
| **Failure types** | Throttling, timeout (>30s), and service unavailability |
| **Retry behavior** | Automatic, immediate, preserves original request content |
| **Chain exhaustion** | RuntimeError with provider count and per-provider error details |
| **Monitoring** | Attempt log tracks all successes and failures for analysis |

### Production Recommendations

1. **Combine both patterns**: Use cost routing for normal operation, fallback for resilience
2. **Monitor and alert**: Track routing decisions and fallback rates in your observability stack
3. **Order by cost**: In fallback chains, put cheapest providers first (any response is acceptable)
4. **Set timeouts**: Always configure timeouts — a hanging request is worse than a fast failure
5. **Log everything**: Attempt logs are essential for post-incident analysis

### Next Steps

- Review tutorial 03 (Model Providers) for basic provider configuration
- Explore the Strands SDK documentation for additional model provider options
- Consider adding circuit breakers for providers with sustained failures